# Job Scout — Phase 3: Ollie reads the traces

**Build → Evaluate → Self-Improve.** Phase 1 built the agent, Phase 2 measured it. This
notebook is about the part that has no SDK: **Ollie**, Opik's built-in assistant, which
reads your traces in the dashboard and answers questions about them.

There is nothing to install and nothing to call. So the notebook's job is to *manufacture
the evidence and hand you the wheel*: it produces two traces of the same search — one with
the source fan-out on, one with it off — prints their durations and links, and then hands
you the exact questions to ask Ollie about them.

**The lesson, stated up front:** an assistant can only find what your instrumentation
records. Phase 3 gave each job source its own span. The same question, asked before and
after that change, gets a different answer — and only one of them is useful.

Companion notes: [`phase3/README.md`](phase3/README.md) ·
[`../docs/ollie.md`](../docs/ollie.md) · [`../docs/optimizing_latency.md`](../docs/optimizing_latency.md)

## 1. Environment check

This notebook needs an **Opik key** — without traces there is nothing for Ollie to read.
Job source keys are optional: the cascade falls back to the committed cache.

In [ ]:
# The cold-import cycle: import schemas before anything pulls in the graph.
from job_scout.graph.schemas import Profile  # noqa: F401

from job_scout.config import get_settings

settings = get_settings()
checks = {
    "Opik tracing (required here)": settings.has_opik,
    "JSearch key": bool(settings.jsearch_api_key.get_secret_value()),
    "Adzuna keys": bool(settings.adzuna_app_id and settings.adzuna_app_key.get_secret_value()),
    "Concurrent fan-out (SCOUT_CONCURRENT_SOURCES)": settings.scout_concurrent_sources,
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else '-- '} {name}")

if not settings.has_opik:
    print("\nNo Opik key: the searches below still run, but nothing is traced and Ollie has nothing to read.")

## 2. What the spans look like now

Phase 3 wraps each live source in its own span. The wrapper is deliberately a **no-op when
Opik is not configured** — `opik.track` otherwise ships spans without a key and answers 401
into your terminal, and this repo promises to stay quiet when keyless.

Note what is spanned: the **query**, not the consumption. The cascade still decides which
results get merged (the `<5` and `<3` thresholds); the spans just make the waiting visible.

In [ ]:
import inspect

from job_scout.tracing import traced_call

print(inspect.getsource(traced_call))

## 3. Time each source on its own

Before the traces, the ground truth: hit each adapter directly and time it. This is the
number Ollie should be able to find for you afterwards — if it cannot, the instrumentation
is at fault, not the assistant.

In [ ]:
import time

from job_scout.tools.jobs_api import AdzunaSource, JSearchSource, RemotiveSource

QUERY, LOCATION, COUNTRY = "data scientist", "Berlin, Germany", "de"

for name, source in [("jsearch", JSearchSource()), ("adzuna", AdzunaSource()), ("remotive", RemotiveSource())]:
    if not getattr(source, "available", True):
        print(f"{name:10s} (no key — skipped)")
        continue
    started = time.monotonic()
    try:
        found = source.fetch(QUERY, LOCATION, COUNTRY, False, 5)
        print(f"{name:10s} {round((time.monotonic() - started) * 1000):6d} ms   {len(found)} jobs")
    except Exception as exc:  # a dead source is an empty source
        print(f"{name:10s} {round((time.monotonic() - started) * 1000):6d} ms   {type(exc).__name__}")

**What we saw on 2026-08-04** (yours will differ — that is the point of measuring):

```
jsearch     15264 ms   0 jobs
adzuna        892 ms   5 jobs
remotive      157 ms   5 jobs
```

15.0s is exactly `JSearchSource`'s timeout. The **primary** source is not slow — it is
timing out, and contributing nothing while it does. Hold that number.

## 4. Two traces: concurrent, then sequential

The same search twice, so you have a pair to compare in the dashboard. Concurrent fan-out
(the Phase 3 default) makes wall time the **slowest single source**; sequential makes it the
**sum**. Both are traced, both get a link printed below.

In [ ]:
import opik

from job_scout.tools.jobs_api import run_search
from job_scout.tracing import configure_opik

configure_opik()


@opik.track(name="ollie-demo-search")
def traced_search(mode: str) -> dict:
    """One search, traced, tagged with the fan-out mode it ran under."""
    started = time.monotonic()
    jobs, used = run_search(QUERY, LOCATION, COUNTRY, False, 10)
    return {"mode": mode, "ms": round((time.monotonic() - started) * 1000), "jobs": len(jobs), "sources_used": used}


results = []
for concurrent in (True, False):
    get_settings.cache_clear()
    import os

    os.environ["SCOUT_CONCURRENT_SOURCES"] = "true" if concurrent else "false"
    results.append(traced_search("concurrent" if concurrent else "sequential"))
    print(results[-1])

opik.flush_tracker()
print("\nflushed — the traces are in the dashboard now")

## 5. Read your own span tree

Before opening the UI, pull the spans back down. Seeing the tree in code first means you
know the right answer when you go on to ask Ollie for it.

In [ ]:
client = opik.Opik()
project = get_settings().opik_project_name
traces = client.search_traces(project_name=project, filter_string='name = "ollie-demo-search"', max_results=2)

for trace in traces:
    total = round((trace.end_time - trace.start_time).total_seconds() * 1000) if trace.end_time else None
    print(f"\ntrace {trace.id}  total {total} ms  output={trace.output}")
    spans = client.search_spans(project_name=project, trace_id=trace.id, max_results=50)
    for span in sorted(spans, key=lambda s: s.start_time):
        ms = round((span.end_time - span.start_time).total_seconds() * 1000) if span.end_time else None
        print(f"    {span.name:22s} {ms:6} ms")

**The pair we measured on 2026-08-04**, straight out of the cell above:

```
concurrent   15368 ms    source.jsearch 15355 · source.adzuna 914 · source.remotive 218
sequential   16228 ms    source.jsearch 15248 · source.adzuna 979
```

Two things are visible here that a single total would hide.

**Sum versus max.** Sequential spends 15248 + 979 = 16227 ms; concurrent spends the slowest
source alone. That is the entire argument for the fan-out, in two span trees.

**The quota trade-off, made concrete.** Sequential has *two* source spans; concurrent has
three. Adzuna returned enough jobs that the cascade never needed Remotive — so sequential
never asked it, while concurrent had already spent the request. Same results either way, but
one of them costs an extra API call. `SCOUT_CONCURRENT_SOURCES=false` buys the quota back
and pays in latency, which is exactly the choice `docs/optimizing_latency.md` describes.

And in both modes: 15 seconds of JSearch, `sources_used = ['adzuna']`.

## 6. Now ask Ollie

Open the project in Opik, open one of the two traces above, and open the Ollie panel.
Ask these in order — the sequence matters more than any single answer:

1. **"This search took N seconds. Which part was slow?"**
   With per-source spans it can name `source.jsearch`. Without them the only honest answer
   is the total you already knew.

2. **"Did the slow source contribute any results?"**
   The answer is in `sources_used` on the trace output. This is the good one: the slow thing
   was also the useless thing, and no amount of staring at the total would have said so.

3. **"The source times out at 15 seconds. What would you change?"**
   Judge the answer against [`../docs/optimizing_latency.md`](../docs/optimizing_latency.md),
   which records what we tried, what worked, and one honest failure. An assistant that
   proposes something we already measured and rejected is worth catching.

4. **Compare the pair.** Point it at the sequential trace and ask how it differs. Sum versus
   max is the whole fan-out argument, visible in two span trees.

### The honest close

Every question above was answerable only because somebody decided a job source deserved its
own span. Ollie did not find the 15-second timeout; the instrumentation did, and Ollie read
it out loud. That is a real and useful thing for a tool to do — and it is not the same thing
as the tool doing your observability for you.

The finding itself is still open, on purpose: lowering a source timeout changes what users
get, not just how fast. Written up in [`../docs/ollie.md`](../docs/ollie.md).